In [ ]:

from pyspark.sql import functions as F

df_payers = spark.read.csv(
    "Files/payers.csv", header=True)

In [ ]:

display(df_payers)




In [ ]:
#TRANSFORMATIONS
df_payers_transformed = (
    df_payers

    # Clean insurance provider name
    .withColumn("NAME", F.trim("NAME"))

    # Clean city name
    .withColumn("CITY", F.trim("CITY"))

    # Standardize state abbreviations
    .withColumn(
        "STATE_HEADQUARTERED",
        F.upper(F.trim("STATE_HEADQUARTERED"))
    )

    # Categorize insurance providers
    .withColumn(
        "PAYER_GROUP",
        F.when(
            F.upper(F.col("NAME")) == "NO_INSURANCE",
            "Uninsured"
        )
        .when(
            F.col("NAME").isin(
                "Medicare", "Medicaid", "Dual Eligible"
            ),
            "Public Coverage"
        )
        .otherwise("Commercial / Other")
    )

    # Keep identifiers as strings
    .withColumn("ZIP", F.col("ZIP").cast("string"))
    .withColumn("PHONE", F.col("PHONE").cast("string"))
)

In [ ]:
display(df_payers_transformed)

In [ ]:
path = spark.conf.get('spark.executorEnv.HC_Goldpath')
display(path)

In [ ]:

(
    df_payers_transformed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "Healthcare_DE_Project.HC_Gold.dbo.DIM_Payers"
    )
)
